In [1]:
from dataclasses import dataclass, asdict
from pathlib import Path
from loguru import logger
import numpy as np
import pandas as pd
import psychrolib

import combined_cooler
from solhycool_visualization.sampling import plot_samples
from phd_visualizations import save_figure

# auto reload modules
%load_ext autoreload
%autoreload 2

psychrolib.SetUnitSystem(psychrolib.SI)
get_Twb = np.vectorize(psychrolib.GetTWetBulbFromRelHum) # Vectorize the function
get_HR = np.vectorize(psychrolib.GetRelHumFromTWetBulb) # Vectorize the function


cc_model = combined_cooler.initialize()

case_study_id: str = "pilot_plant_200kW"
base_output_path: Path = Path("../results/model_inputs_sampling") / case_study_id
base_output_path.mkdir(parents=True, exist_ok=True)

# Parameters



Authorization required, but no authorization protocol specified



# DC

In [21]:
# New approach, directly generate the samples given ranges and a resolution for the inputs

@dataclass
class ModelInputsRange:
    Tamb: tuple[float, float] = (3., 50.)
    deltaTamb_Tin: tuple[float, float] = (3., 30.)
    Tdc_in: tuple[float, float] = (25., 45.)
    qdc: tuple[float, float] = (6., 24.)
    wdc: tuple[float, float] = (11., 99.18)

@dataclass
class NumInputUpdates:
    Tamb: int = 7
    deltaTamb_Tin: int = 7
    qdc: int = 7
    wdc: int = 6

@dataclass
class ModelInputs:
    Tamb: np.ndarray[float]
    Tdc_in: np.ndarray[float]
    qdc: np.ndarray[float]
    wdc: np.ndarray[float]

    @classmethod
    def initialize(cls, num_input_updates: NumInputUpdates = NumInputUpdates(), inputs_range: ModelInputsRange = ModelInputsRange()) -> "ModelInputs":
        # Generate base temperature combinations
        Tamb = np.linspace(inputs_range.Tamb[0], inputs_range.Tamb[1]-inputs_range.deltaTamb_Tin[0], num_input_updates.Tamb)
        deltaTamb_Tin = np.linspace(inputs_range.deltaTamb_Tin[0], inputs_range.deltaTamb_Tin[1], num_input_updates.deltaTamb_Tin)

        Tamb_grid, deltaT_grid = np.meshgrid(Tamb, deltaTamb_Tin, indexing='ij')
        Tdc_in_grid = Tamb_grid + deltaT_grid

        # Mask out invalid Tdc_in
        mask = (Tdc_in_grid <= inputs_range.Tdc_in[1]) & (Tdc_in_grid >= inputs_range.Tdc_in[0])
        Tamb_valid = Tamb_grid[mask]
        Tdc_in_valid = Tdc_in_grid[mask]

        # Generate qdc and wdc ranges
        qdc_vals = np.linspace(inputs_range.qdc[0], inputs_range.qdc[1], num_input_updates.qdc)
        wdc_vals = np.linspace(inputs_range.wdc[0], inputs_range.wdc[1], num_input_updates.wdc)

        # Create full meshgrid over all valid (Tamb, Tdc_in) × (qdc, wdc)
        Tamb_final, qdc_final, wdc_final = np.meshgrid(Tamb_valid, qdc_vals, wdc_vals, indexing='ij')
        Tdc_in_final, _, _ = np.meshgrid(Tdc_in_valid, qdc_vals, wdc_vals, indexing='ij')

        # Flatten everything
        return cls(
            Tamb = Tamb_final.ravel(),
            Tdc_in = Tdc_in_final.ravel(),
            qdc = qdc_final.ravel(),
            wdc = wdc_final.ravel()
        )

# 1. Define problem
output_path = base_output_path / "dc_in.csv"
df = pd.DataFrame(asdict(ModelInputs.initialize()))

display(df)

df.to_csv(output_path)
logger.info(f"{len(df)} samples generated and saved at {output_path}")


,Tamb,Tdc_in,qdc,wdc
0,3.000000,28.500000,6.0,11.000
1,3.000000,28.500000,6.0,28.636
2,3.000000,28.500000,6.0,46.272
3,3.000000,28.500000,6.0,63.908
4,3.000000,28.500000,6.0,81.544
...,...,...,...,...
793,39.666667,42.666667,24.0,28.636
794,39.666667,42.666667,24.0,46.272
795,39.666667,42.666667,24.0,63.908
796,39.666667,42.666667,24.0,81.544


2025-06-14 19:07:26.095 | INFO     | __main__:<module>:62 - 798 samples generated and saved at ../results/model_inputs_sampling/pilot_plant_200kW/dc_in.csv


In [22]:
rename_dict = {
    "Tamb": "T<sub>amb</sub>",
    "Tdc_in": "T<sub>dc,in</sub>",
    "qdc": "q<sub>dc</sub>",
    "wdc": "w<sub>dc</sub>"
}
df.rename(columns=rename_dict, inplace=True)

fig = plot_samples(df, Ncols=len(df.columns))
fig.update_layout(width=800)

save_figure(fig, figure_path=base_output_path, figure_name="viz_samples_dc", formats=["png", "html"], scale=3)

fig


2025-06-14 19:07:29.727 | INFO     | phd_visualizations:save_figure:42 - Figure saved in ../results/model_inputs_sampling/pilot_plant_200kW/viz_samples_dc.png
2025-06-14 19:07:29.731 | INFO     | phd_visualizations:save_figure:42 - Figure saved in ../results/model_inputs_sampling/pilot_plant_200kW/viz_samples_dc.html


# WCT

In [23]:
# New approach, directly generate the samples given ranges and a resolution for the inputs

def get_temperature_and_humidity_values(Twb: float, Tamb_max: float = 50, n_samples_max: int = 10, deltaTmin: float = 1.0 ) -> tuple[np.ndarray, np.ndarray]:
    """ 
    Generate temperature and humidity values based on the wet bulb temperature (Twb) and maximum ambient temperature (Tamb_max).
    The function calculates the ambient temperature (Tamb) and relative humidity (HR) values based on the given Twb.
    The function returns the valid Tamb and HR values that are within the specified range.
    """
    if Twb > Tamb_max:
        raise ValueError(f"Tamb_max ({Tamb_max:.2f})> Tw ({Twb:.2f}) °C")
    n_samples = min(int((Tamb_max - Twb) / deltaTmin), n_samples_max)
    
    Tamb = np.linspace(Twb, Tamb_max, n_samples)
    HR = np.minimum(get_HR(Tamb, np.full(n_samples,Twb), np.full(n_samples, 101325)), 0.999) 
    # Filter out invalid values
    Twb_ = get_Twb(Tamb, HR, np.full(len(Tamb), 101325))
    
    return Tamb[np.abs(Twb_ - Twb) < 1e-1], HR[np.abs(Twb_ - Twb) < 1e-1]

@dataclass
class ModelInputsRange:
    Twb: tuple[float, float] = (3., 40.)
    deltaTwb_Tin: tuple[float, float] = (3., 30.)
    Twct_in: tuple[float, float] = (25., 45.)
    qwct: tuple[float, float] = (6., 24.)
    wwct: tuple[float, float] = (21.0, 93.4161)
    mw_ma_ratio: tuple[float, float] = (0.5, 5.)

@dataclass
class NumInputUpdates:
    Twb: int = 5
    deltaTwb_Tin: int = 5
    qwct: int = 5
    mw_ma_ratio: int = 6

@dataclass
class ModelInputs:
    Tamb: np.ndarray[float]
    HR: np.ndarray[float]
    Twct_in: np.ndarray[float]
    qwct: np.ndarray[float]
    wwct: np.ndarray[float]

    @classmethod
    def initialize(cls, num_input_updates: NumInputUpdates = NumInputUpdates(), inputs_range: ModelInputsRange = ModelInputsRange()) -> "ModelInputs":
        # Generate base temperature combinations
        Twb_grid, deltaT_grid = np.meshgrid(
            np.linspace(inputs_range.Twb[0], inputs_range.Twb[1], num_input_updates.Twb), 
            np.linspace(inputs_range.deltaTwb_Tin[0], inputs_range.deltaTwb_Tin[1], num_input_updates.deltaTwb_Tin), 
            indexing='ij'
        )
        Twct_in_grid = Twb_grid + deltaT_grid
        # Mask out invalid Twct_in
        mask = (Twct_in_grid <= inputs_range.Twct_in[1]) & (Twct_in_grid >= inputs_range.Twct_in[0])
        Twb_valid = Twb_grid[mask]
        Twct_in_valid = Twct_in_grid[mask]
        # Decompose Twb into Tamb and HR
        Twct_in_flat = []
        Tamb_flat = []
        HR_flat = []
        for Twb, Twct_in in zip(Twb_valid.ravel(), Twct_in_valid.ravel()):
            Tamb, HR = get_temperature_and_humidity_values(Twb)
            Tamb_flat.extend(Tamb)
            HR_flat.extend(HR*100)
            Twct_in_flat.extend( [Twct_in]*len(Tamb) )

        # # Generate base qwct and mw_ma_ratio combinations
        qwct_grid, mw_ma_ratio_grid = np.meshgrid(
            np.linspace(inputs_range.qwct[0], inputs_range.qwct[1], num_input_updates.qwct), 
            np.linspace(inputs_range.mw_ma_ratio[0], inputs_range.mw_ma_ratio[1], num_input_updates.mw_ma_ratio), 
            indexing='ij'
        )
        qwct_flat = qwct_grid.ravel()
        mw_ma_ratio_flat = mw_ma_ratio_grid.ravel()
        
        wwct, valid = zip(*[
            cc_model.air_mass_flow_rate_to_fan_speed( qwct/3.6 / mw_ma_ratio, nargout=2 ) # 1/mw/ma [kg/s / kg/s] * mw [kg/s: m3/h -> 1000 kg/m³ * 1/3600 h/s] 
            # (np.random.rand(1) * 90, True)
            for qwct, mw_ma_ratio in zip(qwct_flat, mw_ma_ratio_flat)
        ])
        
        mask = np.array(valid) # np.array(valid).reshape(qwct_grid.shape) 
        qwct_valid = np.array(qwct_flat)[mask]
        wwct_valid = np.array(wwct)[mask]

        # Create full meshgrid over all valid (Tamb, HR, Twct_in) × (qwct, wwct)
        Tamb_final, qwct_final = np.meshgrid(Tamb_flat, qwct_valid, indexing='ij')
        _, wwct_final = np.meshgrid(Tamb_flat, wwct_valid, indexing='ij')
        HR_final, _ = np.meshgrid(HR_flat, qwct_valid, indexing='ij')
        Twct_in_final, _ = np.meshgrid(Twct_in_flat, qwct_valid, indexing='ij')

        # Flatten everything
        return cls(
            Tamb = Tamb_final.ravel(),
            HR = HR_final.ravel(),
            Twct_in = Twct_in_final.ravel(),
            qwct = qwct_final.ravel(),
            wwct = wwct_final.ravel()
        )

# 1. Define problem
output_path = base_output_path / "wct_in.csv"

model_inputs = ModelInputs.initialize()

# From model inputs, compute Twb, mw/ma
ma_kgs = np.array(cc_model.fan_speed_to_air_mass_flow_rate_fit(model_inputs.wwct)[0])

additional_inputs_dict = {
    "Twb": np.array(
        get_Twb(model_inputs.Tamb, model_inputs.HR/100, np.full(len(model_inputs.Tamb), 101325))
    ),
    "mw_ma_ratio": np.array([model_inputs.qwct / (ma_kgs * 3.6)]).reshape(-1)
}

output_dict = {
    **asdict(model_inputs),
    **additional_inputs_dict,
}
df = pd.DataFrame(output_dict)

display(df)

df.to_csv(output_path)
logger.info(f"{len(df)} samples generated and saved at {output_path}")


,Tamb,HR,Twct_in,qwct,wwct,Twb,mw_ma_ratio
0,3.0,99.900000,26.25,6.0,62.012191,2.993384,0.500000
1,3.0,99.900000,26.25,6.0,24.364216,2.993384,1.400000
2,3.0,99.900000,26.25,10.5,38.035240,2.993384,1.400000
3,3.0,99.900000,26.25,10.5,25.478046,2.993384,2.300000
4,3.0,99.900000,26.25,10.5,21.000000,2.993384,3.067915
...,...,...,...,...,...,...,...
1507,50.0,54.625129,43.00,19.5,22.847171,39.999675,5.000000
1508,50.0,54.625129,43.00,24.0,52.757668,39.999675,2.300000
1509,50.0,54.625129,43.00,24.0,38.035240,39.999675,3.200000
1510,50.0,54.625129,43.00,24.0,30.783539,39.999675,4.100000


2025-06-14 19:07:35.931 | INFO     | __main__:<module>:125 - 1512 samples generated and saved at ../results/model_inputs_sampling/pilot_plant_200kW/wct_in.csv


In [24]:
rename_dict = {
    "Tamb": "T<sub>amb</sub>",
    "Twct_in": "T<sub>wct,in</sub>",
    "qwct": "q<sub>wct</sub>",
    "wwct": "ω<sub>wct</sub>",
    "Twb": "T<sub>wb</sub>",
    "mw_ma_ratio": "ṁ<sub>w</sub>/ṁ<sub>ma</sub>"
}
df.rename(columns=rename_dict, inplace=True)

fig = plot_samples(df, Ncols=len(df.columns))
fig.update_layout(width=1000)

save_figure(fig, figure_path=base_output_path, figure_name="viz_samples_wct", formats=["png", "html"], scale=2)

fig


2025-06-14 19:07:39.333 | INFO     | phd_visualizations:save_figure:42 - Figure saved in ../results/model_inputs_sampling/pilot_plant_200kW/viz_samples_wct.png
2025-06-14 19:07:39.338 | INFO     | phd_visualizations:save_figure:42 - Figure saved in ../results/model_inputs_sampling/pilot_plant_200kW/viz_samples_wct.html


# Old

## DC

In [11]:
from SALib.sample import sobol

N: int = 256


In [ ]:
@dataclass
class ModelInputsRange:
    Tamb: tuple[float, float] = (0.1, 50.)
    Tdc_in: tuple[float, float] = (25., 45.)
    qdc: tuple[float, float] = (6., 24.)
    wdc: tuple[float, float] = (11., 99.18)
    
# 1. Define problem
output_path = base_output_path / "dc.csv"
mip = ModelInputsRange()

var_ids = list(asdict(mip).keys())

problem = {
    'num_vars': len(asdict(mip)),
    'names': var_ids,
    'bounds': list(asdict(mip).values()),
    'outputs': ['Tdc_out']
}

# 2. Generate samples
param_values = sobol.sample(problem, N=N, calc_second_order=True)

df = pd.DataFrame(param_values, columns=var_ids)
len0 = len(df)

# Filter out invalid points
# Tin > Tamb
df = df[df["Tdc_in"] < df["Tamb"]]

logger.info(f"After filtering, removed {len0-len(df)} rows, from {len0} to {len(df)}")
display(df)
df.to_csv(output_path)
logger.info(f"Samples saved at {output_path}")

fig = plot_samples(df, var_ids, Ncols=len(var_ids))
display(fig)
save_figure(fig, figure_path=base_output_path, figure_name="viz_samples_dc", formats=["png"])


2025-04-21 09:10:40.648 | INFO     | __main__:<module>:31 - After filtering, removed 1286 rows, from 2560 to 1274


,Tamb,Tdc_in,qdc,wdc
0,40.364071,15.911473,17.965287,26.752296
2,40.364071,9.434187,17.965287,26.752296
3,40.364071,15.911473,17.407316,26.752296
4,40.364071,15.911473,17.965287,52.662090
5,40.364071,9.434187,17.407316,52.662090
...,...,...,...,...
2552,40.500587,25.064700,12.182225,89.453014
2555,40.500587,25.064700,13.367585,38.546102
2557,39.025198,25.064700,12.182225,38.546102
2558,39.025198,25.064700,13.367585,89.453014


2025-04-21 09:10:40.673 | INFO     | __main__:<module>:34 - Samples saved at ../results/model_inputs_sampling/pilot_plant_200kW/dc.csv


2025-04-21 09:10:42.483 | INFO     | phd_visualizations:save_figure:42 - Figure saved in ../results/model_inputs_sampling/pilot_plant_200kW/viz_samples_dc.png


## WCT

In [ ]:
@dataclass
class ModelInputsRange:
    Tamb: tuple[float, float] = (0.1, 50.)
    HR: tuple[float, float] = (10.33, 89.25)
    Twct_in: tuple[float, float] = (25., 45.)
    qwct: tuple[float, float] = (6., 24.)
    wwct: tuple[float, float] = (21.0, 93.4161)

# 1. Define problem
output_path = base_output_path / "wct.csv"
mip = ModelInputsRange()
var_ids = list(asdict(mip).keys())

problem = {
    'num_vars': len(asdict(mip)),
    'names': var_ids,
    'bounds': list(asdict(mip).values()),
    'outputs': ['Twct_out']
}

# 2. Generate samples
param_values = sobol.sample(problem, N=N, calc_second_order=True)

df = pd.DataFrame(param_values, columns=var_ids)
len0 = len(df)

# Filter out invalid points
# Tin > Twb
df = df[ df["Twct_in"] > get_Twb(df["Tamb"], df["HR"]/100, np.full(len(df), 101325)) ]

logger.info(f"After filtering, removed {len0-len(df)} rows, from {len0} to {len(df)}")
display(df)
df.to_csv(output_path)
logger.info(f"Samples saved at {output_path}")

fig = plot_samples(df, var_ids, Ncols=len(var_ids))
display(fig)
save_figure(fig, figure_path=base_output_path, figure_name="viz_samples_wct", formats=["png"])
